# MIMIC-IV — Data Loading & Cohort Construction




## 1. Imports & Config

In [ ]:
import os
import pandas as pd
import numpy as np

PATH = "YOUR PATH HERE"
BASE_DIR = PATH
HOSP_DIR = os.path.join(BASE_DIR, "hosp")
ICU_DIR  = os.path.join(BASE_DIR, "icu")

# Sanity check
print(os.path.exists(HOSP_DIR), os.path.exists(ICU_DIR))

True True


Getting rid of gzip

In [43]:
def csv_path(directory, filename):
    gz    = os.path.join(directory, filename + ".gz")
    plain = os.path.join(directory, filename)
    if os.path.exists(gz):
        return gz
    elif os.path.exists(plain):
        return plain
    raise FileNotFoundError(f"{filename} not found in {directory}")

---
## 2. Patients

One row per patient. Gives us gender and age

In [44]:
patients = pd.read_csv(
    csv_path(HOSP_DIR, "patients.csv"),
    usecols=["subject_id", "gender", "anchor_age", "dod"]
)

patients["gender"]    = patients["gender"].map({"M": 0, "F": 1})
patients["dod_known"] = patients["dod"].notna().astype(int)
patients = patients.drop(columns=["dod"])

print(f"{len(patients)} patients")
patients.head(3)

100 patients


,subject_id,gender,anchor_age,dod_known
0,10014729,1,21,0
1,10003400,1,72,1
2,10002428,1,80,0


## Admissions data set process

Admissions contains a lot of essential information such as insurance category, race and the expiry binary (showing deaths in hospital). I have dropped Uknown, categorised insurance as a binary ( medicare and medicaid are federal plans for disadvantaged people: seniors, low income individuals and disabilities so Other would indicate private/ employer insurance or self paying meaning in general a wealthier patient.)

There is also elective vs emergency admissions, unsure wether to keep this but seems like a important feature. 

In [57]:
admissions = pd.read_csv(
    csv_path(HOSP_DIR, "admissions.csv"),
    usecols=["subject_id", "hadm_id", "admittime", "dischtime",
             "admission_type", "insurance", "marital_status",
             "race", "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime"]
)

# This is HOSPITALE LOS not ICU
admissions["los_hospital_hrs"] = (
    (admissions["dischtime"] - admissions["admittime"])
    .dt.total_seconds() / 3600
).round(2)

# Emergency vs elective
emergency_types = {
    "AMBULATORY OBSERVATION", "DIRECT EMER.", "DIRECT OBSERVATION",
    "EU OBSERVATION", "EW EMER.", "OBSERVATION ADMIT", "URGENT"
}
admissions["is_emergency"] = admissions["admission_type"].isin(emergency_types).astype(int)


race_map = {
    "WHITE": "WHITE",
    "BLACK/AFRICAN AMERICAN": "BLACK",
    "HISPANIC/LATINO": "HISPANIC",
    "ASIAN": "ASIAN",
    "AMERICAN INDIAN/ALASKA NATIVE": "OTHER",
    "OTHER": "OTHER",
}
admissions["race_group"] = (
    admissions["race"].str.upper().str.strip()
    .map(race_map)
)
admissions = admissions[admissions["race_group"].notna()].reset_index(drop=True)

admissions["insurance_group"] = (
    admissions["insurance"]
    .map({"Medicare": "government", "Medicaid": "government"})
    .fillna("private")
)

admissions = admissions.drop(columns=["admittime", "dischtime", "admission_type", "race", "insurance"])

print(f"{len(admissions)} admissions | Mortality rate: {admissions['hospital_expire_flag'].mean():.2%}")
print("\nRace distribution:")
print(admissions["race_group"].value_counts())
admissions.head(3)

222 admissions | Mortality rate: 4.50%

Race distribution:
race_group
WHITE    170
BLACK     48
OTHER      4
Name: count, dtype: int64


,subject_id,hadm_id,marital_status,hospital_expire_flag,los_hospital_hrs,is_emergency,race_group,insurance_group
0,10018081,23983182,MARRIED,0,137.55,1,WHITE,government
1,10031404,21606243,WIDOWED,0,50.18,1,WHITE,private
2,10005817,20626031,MARRIED,0,205.35,1,WHITE,government


---
## 4. ICU Stays

First ICU Stay only as if there is a second stay that means they survived the first one anyway and it also keeps the final df cleaner with each row representing a single patient admission rather than the same patient repeatedly

In [ ]:
icustays = pd.read_csv(
    csv_path(ICU_DIR, "icustays.csv"),
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"],
    parse_dates=["intime", "outtime"]
)

icustays = icustays.sort_values("intime").groupby("hadm_id").first().reset_index()
icustays["long_icu_stay"] = (icustays["los"] > 3).astype(int)

print(f"{len(icustays)} ICU stays")
icustays.head(3)

128 ICU stays


,hadm_id,subject_id,stay_id,intime,outtime,los,long_icu_stay
0,20044587,10023771,33177122,2113-08-25 09:32:41,2113-08-27 16:27:53,2.288333,0
1,20199380,10005909,36496303,2144-10-29 23:09:03,2144-11-02 15:24:29,3.677384,1
2,20214994,10003400,32128372,2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,1


---
## 5. Comorbidities (from ICD diagnosis codes)

Each admission has a list of ICD diagnosis codes (both ICD-9 and ICD-10 depending on year). We map these to 8 binary flags for common conditions that are well-known mortality predictors.

The raw table is long format (one row per diagnosis per admission) — we pivot it to one row per admission with binary columns.

In [47]:
diagnoses = pd.read_csv(
    csv_path(HOSP_DIR, "diagnoses_icd.csv"),
    usecols=["hadm_id", "icd_code"]
)

comorbidity_map = {
    "250": "diabetes",    "E11": "diabetes",    "E10": "diabetes",
    "401": "hypertension","402": "hypertension","403": "hypertension",
    "I10": "hypertension","I11": "hypertension",
    "410": "mi",          "411": "mi",          "412": "mi",
    "I21": "mi",          "I22": "mi",
    "428": "heart_failure","I50": "heart_failure",
    "490": "copd",        "491": "copd",        "492": "copd",
    "496": "copd",        "J44": "copd",        "J43": "copd",
    "585": "renal_failure","586": "renal_failure",
    "N18": "renal_failure","N19": "renal_failure",
    "570": "liver_disease","571": "liver_disease",
    "K70": "liver_disease","K74": "liver_disease",
    "140": "cancer",      "141": "cancer",      "142": "cancer",
    "C0":  "cancer",      "C1":  "cancer",      "C2":  "cancer",
    "C3":  "cancer",      "C4":  "cancer",
}

def map_code(code):
    for prefix, name in comorbidity_map.items():
        if str(code).startswith(prefix):
            return name
    return None

diagnoses["icd_code"]    = diagnoses["icd_code"].str.strip().str.upper()
diagnoses["comorbidity"] = diagnoses["icd_code"].apply(map_code)
diagnoses = diagnoses[diagnoses["comorbidity"].notna()]

comorbidities = (
    diagnoses.groupby(["hadm_id", "comorbidity"]).size()
    .gt(0).astype(int)
    .unstack("comorbidity").fillna(0).astype(int)
    .reset_index()
)
comorbidities.columns = ["hadm_id"] + [f"cm_{c}" for c in comorbidities.columns[1:]]

print(f"{len(comorbidities)} admissions with comorbidity data")
comorbidities.head(3)

223 admissions with comorbidity data


,hadm_id,cm_cancer,cm_copd,cm_diabetes,cm_heart_failure,cm_hypertension,cm_liver_disease,cm_mi,cm_renal_failure
0,20044587,0,0,0,0,1,0,0,0
1,20093566,0,0,0,0,0,1,0,1
2,20192635,0,0,1,0,1,0,0,1


---
## 6. Lab Results (first 24h of ICU stay)

`labevents` is a massive long-format table — millions of rows, one per lab test per patient per timepoint. We:
1. Filter to 14 clinically relevant labs (creatinine, lactate, glucose etc.)
2. Keep only measurements from the **first 24 hours** of ICU admission (no future leakage)
3. Pivot to one row per stay with mean values

The 24h window is standard in MIMIC mortality prediction literature.

In [48]:
LAB_ITEMS = {
    50912: "creatinine",
    50971: "potassium",
    50983: "sodium",
    50902: "chloride",
    50882: "bicarbonate",
    51006: "urea_nitrogen",
    51221: "hematocrit",
    51222: "hemoglobin",
    51265: "platelets",
    51301: "wbc",
    50868: "anion_gap",
    50931: "glucose",
    50813: "lactate",
    50820: "ph",
}

labevents = pd.read_csv(
    csv_path(HOSP_DIR, "labevents.csv"),
    usecols=["hadm_id", "itemid", "charttime", "valuenum"],
    parse_dates=["charttime"],
    low_memory=False
)

labevents = labevents[labevents["itemid"].isin(LAB_ITEMS)].copy()
labevents["lab_name"] = labevents["itemid"].map(LAB_ITEMS)

# Join ICU admission time and filter to first 24h
labevents = labevents.merge(icustays[["hadm_id", "stay_id", "intime"]], on="hadm_id")
labevents = labevents[
    (labevents["charttime"] >= labevents["intime"]) &
    (labevents["charttime"] <= labevents["intime"] + pd.Timedelta(hours=24))
]

labs = (
    labevents.groupby(["stay_id", "lab_name"])["valuenum"]
    .mean().unstack("lab_name").reset_index()
)
labs.columns = ["stay_id"] + [f"lab_{c}" for c in labs.columns[1:]]

print(f"{len(labs)} stays with lab data, {labs.shape[1]-1} lab features")
labs.head(3)

128 stays with lab data, 14 lab features


,stay_id,lab_anion_gap,lab_bicarbonate,lab_chloride,lab_creatinine,lab_glucose,lab_hematocrit,lab_hemoglobin,lab_lactate,lab_ph,lab_platelets,lab_potassium,lab_sodium,lab_urea_nitrogen,lab_wbc
0,30057454,16.0,27.0,96.25,1.75,134.0,41.2,13.75,0.6,7.43,214.0,3.825,137.5,42.0,15.25
1,30101877,18.0,23.5,99.50,0.90,184.5,29.6,10.15,NaN,NaN,453.0,4.700,136.0,22.0,20.95
2,30425410,15.0,26.0,101.00,0.50,151.0,31.4,10.00,1.5,7.45,448.0,3.900,138.0,20.0,8.50


---
## 7. Vitals / Chart Events (first 24h of ICU stay)

`chartevents` is the largest table — continuous bedside monitoring data. Same approach as labs: filter to our vital signs of interest, restrict to the first 24h, and pivot wide.

We also compute `gcs_total` (Glasgow Coma Scale) by summing the three component scores — it's a standard neurological severity measure.

In [49]:
CHART_ITEMS = {
    220045: "heart_rate",
    220179: "sbp",
    220180: "dbp",
    220181: "map",
    220210: "resp_rate",
    220277: "spo2",
    223761: "temperature_f",
    220739: "gcs_eye",
    223900: "gcs_verbal",
    223901: "gcs_motor",
}

stay_intime = icustays.set_index("stay_id")["intime"].to_dict()
chunks = []

for chunk in pd.read_csv(
    csv_path(ICU_DIR, "chartevents.csv"),
    usecols=["stay_id", "itemid", "charttime", "valuenum"],
    parse_dates=["charttime"],
    chunksize=500_000,
    low_memory=False
):
    chunk = chunk[
        chunk["itemid"].isin(CHART_ITEMS) &
        chunk["valuenum"].notna() &
        chunk["stay_id"].isin(stay_intime)
    ].copy()
    chunk["intime"] = chunk["stay_id"].map(stay_intime)
    chunk = chunk[
        (chunk["charttime"] >= chunk["intime"]) &
        (chunk["charttime"] <= chunk["intime"] + pd.Timedelta(hours=24))
    ]
    chunks.append(chunk[["stay_id", "itemid", "valuenum"]])

chartevents = pd.concat(chunks, ignore_index=True)
chartevents["vital_name"] = chartevents["itemid"].map(CHART_ITEMS)

# GCS total
gcs_cols = ["gcs_eye", "gcs_verbal", "gcs_motor"]
gcs_total = (
    chartevents[chartevents["vital_name"].isin(gcs_cols)]
    .groupby(["stay_id", "vital_name"])["valuenum"].mean()
    .unstack().sum(axis=1).reset_index()
    .rename(columns={0: "gcs_total"})
)

charts = (
    chartevents.groupby(["stay_id", "vital_name"])["valuenum"]
    .mean().unstack("vital_name").reset_index()
)
charts.columns = ["stay_id"] + [f"vital_{c}" for c in charts.columns[1:]]
charts = charts.merge(gcs_total, on="stay_id", how="left")

print(f"{len(charts)} stays with vital data, {charts.shape[1]-1} vital features")
charts.head(3)

128 stays with vital data, 11 vital features


,stay_id,vital_dbp,vital_gcs_eye,vital_gcs_motor,vital_gcs_verbal,vital_heart_rate,vital_map,vital_resp_rate,vital_sbp,vital_spo2,vital_temperature_f,gcs_total
0,30057454,54.666667,4.000000,6.000000,5.0,108.533333,60.333333,18.266667,79.333333,92.366667,98.057143,15.000000
1,30101877,69.750000,3.615385,5.615385,1.0,94.875000,85.625000,20.541667,137.916667,100.000000,100.071429,10.230769
2,30425410,69.440000,3.857143,6.000000,1.0,95.576923,79.680000,19.307692,115.360000,99.259259,98.014286,10.857143


---
## 8. Build the Cohort

Join out from ICUstay combining all dfs into our final cohort df

In [ ]:
cohort = (
    icustays
    .merge(admissions,    on=["subject_id", "hadm_id"])
    .merge(patients,      on="subject_id")
    .merge(comorbidities, on="hadm_id",  how="left")
    .merge(labs,          on="stay_id",  how="left")
    .merge(charts,        on="stay_id",  how="left")
)

cm_cols = [c for c in cohort.columns if c.startswith("cm_")]
cohort[cm_cols] = cohort[cm_cols].fillna(0)

# Drop raw timestamps 
cohort = cohort.drop(columns=["intime", "outtime"])

print(f"Cohort: {len(cohort)} stays, {cohort.shape[1]} columns")
print(f"Mortality rate: {cohort['hospital_expire_flag'].mean():.2%}")

Cohort: 128 stays, 47 columns
Mortality rate: 11.72%


---
## 9. Missing Check

Checking number of missings for the features


In [ ]:
missing = cohort.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
missing *= 100
print(missing.to_string())

Columns with missing values:
lab_ph                 39.06250
lab_lactate            39.06250
vital_temperature_f     8.59375
vital_dbp               7.81250
marital_status          7.81250
vital_map               7.81250
vital_sbp               7.81250
lab_platelets           3.12500
lab_hemoglobin          2.34375
lab_hematocrit          2.34375
lab_wbc                 2.34375
lab_urea_nitrogen       0.78125
lab_anion_gap           0.78125
lab_bicarbonate         0.78125
lab_glucose             0.78125
lab_creatinine          0.78125


---
## 10. Quick Sanity Checks



In [53]:
print("Race group distribution:")
print(cohort["race_group"].value_counts())
print()
print("Insurance group distribution:")
print(cohort["insurance_group"].value_counts())
print()
print("Mortality by race group:")
print(cohort.groupby("race_group")["hospital_expire_flag"].agg(["mean", "count"]).round(3))

Race group distribution:
race_group
WHITE      83
BLACK      15
OTHER      15
UNKNOWN    15
Name: count, dtype: int64

Insurance group distribution:
insurance_group
private       68
government    60
Name: count, dtype: int64

Mortality by race group:
             mean  count
race_group              
BLACK       0.067     15
OTHER       0.000     15
UNKNOWN     0.333     15
WHITE       0.108     83


In [54]:
cohort.head()

,hadm_id,subject_id,stay_id,los,long_icu_stay,marital_status,hospital_expire_flag,los_hospital_hrs,is_emergency,race_group,...,vital_gcs_eye,vital_gcs_motor,vital_gcs_verbal,vital_heart_rate,vital_map,vital_resp_rate,vital_sbp,vital_spo2,vital_temperature_f,gcs_total
0,20044587,10023771,33177122,2.288333,0,MARRIED,0,127.00,0,WHITE,...,1.75,2.250000,2.000000,78.090909,NaN,15.837209,NaN,99.840909,98.742857,6.000000
1,20199380,10005909,36496303,3.677384,1,MARRIED,0,112.05,1,WHITE,...,4.00,6.000000,5.000000,102.730769,91.043478,18.461538,126.304348,93.692308,98.200000,15.000000
2,20214994,10003400,32128372,12.911308,1,MARRIED,0,557.75,1,BLACK,...,2.00,2.333333,1.000000,115.297872,75.418182,18.933333,100.600000,99.250000,96.680000,5.333333
3,20291550,10008454,31959184,4.983889,1,SINGLE,0,249.37,1,WHITE,...,4.00,6.000000,5.000000,112.120000,86.000000,19.160000,137.000000,95.480000,98.500000,15.000000
4,20297618,10019385,39268883,1.313287,0,MARRIED,0,233.28,1,WHITE,...,2.50,4.833333,2.333333,64.600000,81.000000,20.200000,133.000000,97.379310,98.188889,9.666667


---
## 11. Save

Save the cohort so we don't have to re-run the loading pipeline every time.

In [55]:
cohort.to_csv("cohort.csv", index=False)
print("Saved to cohort.csv")

Saved to cohort.csv
